In [ ]:
%cd /kaggle/working
!git clone https://github.com/shujaabbasn/ATML.git
%cd /kaggle/working/ATML/pa4
!python -m pip install -q -r requirements.txt
!nvidia-smi

In [ ]:
# staff data repo (scripts + bundled files), then merge anything attached from earlier sessions
!git clone -q https://github.com/AbDu11aHHH/atml-pa4-datasets.git data_repo
!python kaggle_runs/attach_inputs.py
# skips files that already exist, so with data_prepare attached this is only a check (C4 streaming is slow otherwise)
!cd data_repo && python scripts/setup_data.py --public && python scripts/check_setup.py

In [ ]:
# attach task2_attention_mqar_run as an input: its 39 finished runs come in through attach_inputs above
# the first three hybrid runs stored a running SUM in fp16, it passed 65504 and went NaN, so only they are redone
!rm -rf results/task2/part_b/hybrid_K16_mixed_seed*
!ls results/task2/part_b

In [ ]:
# retrains only the three hybrid seeds (state now stored as a running mean in fp16), the other runs are not touched
!python -m task2_attention.mqar --arch hybrid

In [ ]:
# forward-only sweep again so its hybrid row uses the same code as the trained models
!python -m task2_attention.mqar --inference_sweep
!python -m task2_attention.mqar --summary

In [ ]:
!zip -r /kaggle/working/task2_hybrid_results.zip results/task2/part_b -x 'results/task2/part_b/train_stream_*.npz' -x 'results/task2/part_b/*/model_update*.pt'